# Two years of data, and not one bit about how long the effect lasts

The panel is beautiful. Two hundred weeks, clean outcomes, a treatment running the whole time
at a steady level. The model has a carryover parameter, the sampler returns a posterior for
it, and that posterior is the prior — because a dose that never changes carries no information
about how quickly its effect decays.

This is the failure that no amount of data fixes and no diagnostic on the fitted model
reports. It is a property of the *design*, and it is checkable before the first observation.

A carryover's weights sum to one, so a dose held constant passes through it unchanged and the
data carry *no* information on the decay. Identification comes from temporal contrast.
`Schedule` builds the patterns; `contrast_score` — mean squared first difference over mean
squared dose, scale-free, `0` for a flat schedule and at most `4` — is the heuristic, and
`fisher_information` on the surface is the model-based verdict.

**The one-forward rule.** Nothing here re-implements the surface. The Jacobian's linear
columns are exactly the design matrix `surface.linearize` returns, and the nonlinear columns
are derivatives of `surface.forward` itself (jax when available with x64, central finite
differences otherwise). Then `FI = JᵀJ / noise_sd²`, and the Laplace posterior covariance
under independent Gaussian priors is `(diag(1/prior_sd²) + FI)⁻¹`.

In [ ]:
import numpy as np

from axiom.core import Unsupported
from axiom.design import (
    DerivativeMethod, FisherInformation, IdentifiabilityRidge, IdentifyingDesign, Pattern, Schedule,
    alternating, constant, contrast_score, design_to_identify, expected_posterior_sd,
    fisher_information, identifiability_ridge, pulse, ramp, random_switchback, ridge_of,
)
from axiom.sim import DosePlan, surface_world
from axiom.surface import Design, GeometricCarryover, HillKernel

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import CRITICAL, caption, compare, dumbbell, lines

enable();  # every axiom result renders itself from here on

In [ ]:
T = 12
schedules: list[Schedule] = [
    constant(T, 50.0),
    pulse(T, 80.0, 20.0, on=2, off=2),
    alternating(T, 80.0, 20.0),
    ramp(T, 0.0, 100.0),
    random_switchback(T, 80.0, 20.0, seed=0),
]
rows = []
for s in schedules:
    p: Pattern = s.pattern
    rows.append([p, f"{s.mean:.1f}", f"{s.total:.1f}", f"{contrast_score(s):.3f}", f"{s.doses[:6]}..."])
table(rows, headers=("pattern", "mean dose", "total", "contrast", "first six doses"))

In [ ]:
fig = lines(
    np.arange(T),
    {s.pattern: s.doses for s in schedules if s.pattern in ("constant", "pulse", "alternating")},
    title="Same treatment, same average dose, different information",
    subtitle="three dose schedules over twelve periods — the top one spends exactly as much as the others",
    x_title="period", y_title="dose",
)
caption(fig, "Nothing distinguishes these schedules in total spend. What distinguishes them "
             "is temporal contrast, and that is the only thing a carryover parameter can be "
             "learned from.")

## A Hill surface with geometric carryover

`axiom.sim.surface_world` builds the surface, its true parameters and a panel; the design math
only needs `world.surface` (a `SupportsForward`) and the data mapping `forward` reads.

In [ ]:
N_UNITS, NOISE_SD = 2, 0.5
truth = {"beta_a": 10.0, "alpha": 5.0, "k_a": 50.0, "s_a": 2.0, "lam_a": 0.5}
world = surface_world(
    n_units=N_UNITS, n_periods=T, treatments=("a",),
    kernels=HillKernel(reference_dose=50.0, amplitude_scale=10.0),
    carryover=GeometricCarryover(max_lag=4), doses=DosePlan(scale=50.0),
    intercept="shared", truth=truth, noise_sd=NOISE_SD, seed=0,
)
method: DerivativeMethod = "finite"

def data_with(schedule: Schedule) -> dict[str, np.ndarray]:
    data = dict(world.data)
    data["a"] = schedule.as_grid(N_UNITS)
    return data

rows = []
for s in schedules:
    fi = fisher_information(world.surface, data_with(s), world.theta, NOISE_SD, method=method)
    assert isinstance(fi, FisherInformation)
    i = fi.index("lam_a")
    rows.append([s.pattern, f"{fi.as_array()[i, i]:.3f}", f"{fi.det:.4g}", str(fi.singular)])
table(rows, headers=("pattern", "info[lam_a]", "det", "singular"))

In [ ]:
info_by_pattern = {}
for sch in schedules:
    fi = fisher_information(world.surface, data_with(sch), world.theta, NOISE_SD, method=method)
    info_by_pattern[f"{sch.pattern}  (contrast {contrast_score(sch):.2f})"] = float(fi.as_array()[fi.index("lam_a"), fi.index("lam_a")])

fig = compare(
    list(info_by_pattern), list(info_by_pattern.values()),
    highlight=min(info_by_pattern, key=info_by_pattern.get),
    value_fmt="{:.0f}",
    title="What each schedule tells you about the carryover",
    subtitle="Fisher information on the decay parameter — same surface, same twelve periods, same noise",
    x_title="information on lam_a",
)
caption(fig, "A pulsed schedule is worth nine times a slow ramp on this parameter, at the "
             "same spend. The constant row is not zero only because the panel starts from "
             "nothing, and the next figure is about what that is worth.")

In [ ]:
lengths = (12, 24, 48, 96)
growth = {"constant dose": [], "pulsed dose": []}
for periods in lengths:
    long_world = surface_world(
        n_units=N_UNITS, n_periods=periods, treatments=("a",),
        kernels=HillKernel(reference_dose=50.0, amplitude_scale=10.0),
        carryover=GeometricCarryover(max_lag=4), doses=DosePlan(scale=50.0),
        intercept="shared", truth=truth, noise_sd=NOISE_SD, seed=0,
    )
    for label, sch in (("constant dose", constant(periods, 50.0)),
                       ("pulsed dose", pulse(periods, 80.0, 20.0, on=2, off=2))):
        data = dict(long_world.data)
        data["a"] = sch.as_grid(N_UNITS)
        fi = fisher_information(long_world.surface, data, long_world.theta, NOISE_SD, method=method)
        growth[label].append(float(fi.as_array()[fi.index("lam_a"), fi.index("lam_a")]))

fig = lines(
    lengths, growth,
    title="The panel that never learns",
    subtitle="information about the decay parameter as the panel gets longer, under two schedules",
    x_title="periods observed", y_title="information on lam_a",
)
caption(fig, f"Constant dose: {growth['constant dose'][0]:.0f} at twelve periods and "
             f"{growth['constant dose'][-1]:.0f} at ninety-six — every bit of it from the "
             f"transient at the start, and none of it from the ninety-five weeks after. "
             f"Running the treatment on a pulse turns the same money into a straight line.")

In [ ]:
flat = fisher_information(world.surface, data_with(constant(T, 50.0)), world.theta, NOISE_SD, method=method)
assert isinstance(flat, FisherInformation)
print("parameters:", flat.parameters, "| round-off columns:", flat.detail.get("round_off_columns"))
print("flat-prior covariance ->", flat.covariance().reason[:90], "...")
print("expected posterior sd (flat prior) ->", type(expected_posterior_sd(None, flat)).__name__)

In [ ]:
pulsed = fisher_information(world.surface, data_with(pulse(T, 80.0, 20.0, on=2, off=2)), world.theta, NOISE_SD, method=method)
assert isinstance(pulsed, FisherInformation)
priors = {"alpha": 5.0, "beta_a": 5.0, "k_a": 20.0, "s_a": 1.0, "lam_a": 0.3}
sd1 = expected_posterior_sd(priors, pulsed)
sd2 = expected_posterior_sd(priors, pulsed + pulsed)  # information adds over independent rows
assert isinstance(sd1, dict) and isinstance(sd2, dict)
table(
    [[name, f"{priors[name]:.2f}", f"{sd1[name]:.4f}", f"{sd2[name]:.4f}"] for name in pulsed.parameters],
    headers=("parameter", "prior sd", "posterior sd", "with twice the rows"),
)

In [ ]:
fig = dumbbell(
    list(pulsed.parameters),
    [priors[name] for name in pulsed.parameters],
    [sd1[name] for name in pulsed.parameters],
    before_label="prior sd", after_label="expected posterior sd",
    title="What the pulsed design is expected to buy",
    subtitle="per parameter, before a single observation is collected",
    x_title="standard deviation",
)
caption(fig, "This is a design calculation, not a fit: no data exists yet. A parameter whose "
             "dot barely moves is one the experiment will not resolve, and the time to find "
             "that out is now.")

## The identifiability ridge

The flattest direction of the correlation-form information matrix is the combination of
parameters the design moves least — for a saturating kernel, the `beta`/`k` equifinality
ridge. Pairwise flat-prior posterior correlations need a nonsingular matrix.

In [ ]:
ridge = identifiability_ridge(world.surface, data_with(pulse(T, 80.0, 20.0, on=2, off=2)), world.theta, NOISE_SD, pairs=(("beta_a", "k_a"),), method=method)
assert isinstance(ridge, IdentifiabilityRidge)
print("direction:", {k: round(v, 3) for k, v in ridge.direction.items()})
print("ridge parameters:", ridge.ridge_parameters, "| condition number:", round(ridge.condition_number, 1))
print("corr(beta_a, k_a) =", round(ridge.correlations[0], 3))
print("from the flat design:", ridge_of(flat).ridge_parameters, "| with pairs ->", type(ridge_of(flat, pairs=(("beta_a", "k_a"),))).__name__)

## Choosing rows to identify one parameter

`design_to_identify` picks `n` candidate rows (replicates allowed) minimizing the target's
expected posterior sd by point exchange. A surface with carryover is handed in as its
`steady_state()`, whose `forward` accepts independent rows.

In [ ]:
steady = world.surface.steady_state()
theta = {k: v for k, v in world.theta.items() if k != "lam_a"}
candidates = Design(treatments=("a",), points=((0.0,), (20.0,), (50.0,), (100.0,), (200.0,)), kind="grid")
out = design_to_identify(steady, candidates, theta, NOISE_SD, target="k_a", n=8,
                         prior_sds={"alpha": 5.0, "beta_a": 5.0, "k_a": 20.0, "s_a": 1.0}, seed=0, method=method)
assert isinstance(out, IdentifyingDesign)
print("chosen doses:", [p[0] for p in out.design.points])
print(f"expected sd of k_a: {out.expected_sd:.3f} (prior 20.0); all: { {k: round(v, 3) for k, v in out.expected_sds.items()} }")
print("one row under flat priors ->", type(design_to_identify(steady, candidates, theta, NOISE_SD, target="k_a", n=1, seed=0, method=method)).__name__)

## What this bought you

The question "will this experiment identify the thing I care about?" answered before the
experiment — from the same `forward()` the likelihood will use, so the answer cannot be about
a different model. Plus the flattest direction of the design named, which is where two
parameters will trade off against each other no matter how much data arrives.